# Phân loại tin tức VietNamNet bằng Semantic KNN

Notebook này gồm **4 nhánh vector hóa / embedding** theo đúng thứ tự bạn muốn thử:

1. **Phần chính**: `AITeamVN/Vietnamese_Embedding`
2. **Phần chính**: `PhoBERT encoder` (`vinai/phobert-base-v2`)
3. **Appendix A**: `BAAI/bge-m3`
4. **Appendix B**: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`

Thiết kế này giúp bạn có thể:
- chạy `Vietnamese_Embedding` và benchmark 4 mô hình phân loại
- chạy `PhoBERT encoder` và benchmark 4 mô hình phân loại
- giữ `bge-m3` và `multilingual` ở cuối notebook dưới dạng code sẵn nhưng mặc định không chạy
- xóa nguyên từng khối section mà không ảnh hưởng các khối khác

| Khối | Nội dung |
|------|----------|
| **Section 0-4** | Setup, load data, EDA, preprocessing, split |
| **Section 5-9** | `Vietnamese_Embedding` + `KNN / LR / SVM / MLP` |
| **Section 10-14** | `PhoBERT encoder` + `KNN / LR / SVM / MLP` |
| **Section 15-18** | Appendix A: `BAAI/bge-m3` (mặc định không chạy) |
| **Section 19-22** | Appendix B: `multilingual mpnet` (mặc định không chạy) |

> **Biểu diễn văn bản**: `title + content` theo chiến lược `head-tail` động theo `max_length`
>
> **Notebook này ưu tiên chạy bằng GPU** để giảm thời gian vector hóa.


---
## Section 0 - Setup

Chạy **mỗi lần** mở notebook.

In [ ]:
import importlib
import os
import re
import json
import time
import pickle
import random
import warnings
from contextlib import contextmanager

_REQUIRED = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'torch': 'torch',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'tqdm': 'tqdm',
    'sklearn': 'scikit-learn',
    'transformers': 'transformers',
    'pyarrow': 'pyarrow',
    'pyvi': 'pyvi',
}
_missing = {pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None}
if _missing:
    raise SystemExit(f'Thiếu thư viện: {sorted(_missing)}')

warnings.filterwarnings('ignore')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from pyvi import ViTokenizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from transformers import AutoTokenizer, AutoModel

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
sns.set_theme(style='whitegrid')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise SystemExit('Notebook này đang được thiết kế ưu tiên GPU. Hiện không phát hiện CUDA/GPU.')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

@contextmanager
def timer(name):
    start = time.time()
    yield
    print(f'[TIME] {name}: {time.time() - start:.2f}s')

def log(msg, tag='INFO'):
    print(f'[{tag}] {msg}')

print('Python :', os.sys.version.split()[0])
print('Device :', device)
print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
NOTEBOOK_DIR = os.getcwd()
DATASET_FOLDER = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'Dataset'))
TEMP_DIR = os.path.join(NOTEBOOK_DIR, 'temp')
RESULTS_DIR = os.path.join(NOTEBOOK_DIR, 'results')
MODEL_DIR = os.path.join(NOTEBOOK_DIR, 'model')
for _d in [TEMP_DIR, RESULTS_DIR, MODEL_DIR]:
    os.makedirs(_d, exist_ok=True)

PROCESSED_DATA_PATH = os.path.join(TEMP_DIR, 'processed_data.pkl')

VI_MODEL_NAME = 'AITeamVN/Vietnamese_Embedding'
VI_MAX_LENGTH = 256
VI_BATCH_SIZE = 8
VI_TOP_K = 7
VI_VOTE_POWER = 2.0
VI_MIN_SIMILARITY = 0.05
VI_TRAIN_EMB_PATH = os.path.join(TEMP_DIR, 'vi_train_embeddings.npz')
VI_TEST_EMB_PATH = os.path.join(TEMP_DIR, 'vi_test_embeddings.npz')
VI_RETRIEVAL_CACHE_PATH = os.path.join(MODEL_DIR, 'vi_retrieval_artifacts.pkl')
VI_LABEL_CONFIG_PATH = os.path.join(MODEL_DIR, 'vi_label_config.json')

PHOBERT_MODEL_NAME = 'vinai/phobert-base-v2'
PHOBERT_MAX_LENGTH = 256
PHOBERT_BATCH_SIZE = 8
PHOBERT_TOP_K = 7
PHOBERT_VOTE_POWER = 2.0
PHOBERT_MIN_SIMILARITY = 0.05
PHOBERT_TRAIN_EMB_PATH = os.path.join(TEMP_DIR, 'phobert_train_embeddings.npz')
PHOBERT_TEST_EMB_PATH = os.path.join(TEMP_DIR, 'phobert_test_embeddings.npz')
PHOBERT_RETRIEVAL_CACHE_PATH = os.path.join(MODEL_DIR, 'phobert_retrieval_artifacts.pkl')
PHOBERT_LABEL_CONFIG_PATH = os.path.join(MODEL_DIR, 'phobert_label_config.json')

BGE_MODEL_NAME = 'BAAI/bge-m3'
BGE_MAX_LENGTH = 256
BGE_BATCH_SIZE = 8
BGE_TOP_K = 7
BGE_VOTE_POWER = 2.0
BGE_MIN_SIMILARITY = 0.05
BGE_TRAIN_EMB_PATH = os.path.join(TEMP_DIR, 'bge_train_embeddings.npz')
BGE_TEST_EMB_PATH = os.path.join(TEMP_DIR, 'bge_test_embeddings.npz')
BGE_RETRIEVAL_CACHE_PATH = os.path.join(MODEL_DIR, 'bge_retrieval_artifacts.pkl')
BGE_LABEL_CONFIG_PATH = os.path.join(MODEL_DIR, 'bge_label_config.json')
RUN_BGE = False

MPNET_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
MPNET_MAX_LENGTH = 256
MPNET_BATCH_SIZE = 16
MPNET_TOP_K = 7
MPNET_VOTE_POWER = 2.0
MPNET_MIN_SIMILARITY = 0.05
MPNET_TRAIN_EMB_PATH = os.path.join(TEMP_DIR, 'mpnet_train_embeddings.npz')
MPNET_TEST_EMB_PATH = os.path.join(TEMP_DIR, 'mpnet_test_embeddings.npz')
MPNET_RETRIEVAL_CACHE_PATH = os.path.join(MODEL_DIR, 'mpnet_retrieval_artifacts.pkl')
MPNET_LABEL_CONFIG_PATH = os.path.join(MODEL_DIR, 'mpnet_label_config.json')
RUN_MPNET = False

TEST_SIZE = 0.15
RANDOM_STATE = 42

LABEL_MAP = {
    'ban-doc': 'Bạn đọc',
    'bao-ve-nguoi-tieu-dung': 'Bảo vệ người tiêu dùng',
    'bat-dong-san': 'Bất động sản',
    'chinh-tri': 'Chính trị',
    'cong-nghe': 'Công nghệ',
    'dan-toc-ton-giao': 'Dân tộc - Tôn giáo',
    'doi-song': 'Đời sống',
    'du-lich': 'Du lịch',
    'giao-duc': 'Giáo dục',
    'kinh-doanh': 'Kinh doanh',
    'oto-xe-may': 'Ô tô - Xe máy',
    'phap-luat': 'Pháp luật',
    'suc-khoe': 'Sức khỏe',
    'the-gioi': 'Thế giới',
    'the-thao': 'Thể thao',
    'thi-truong-tieu-dung': 'Thị trường tiêu dùng',
    'thoi-su': 'Thời sự',
    'tuan-viet-nam': 'Tuần Việt Nam',
    'van-hoa-giai-tri': 'Văn hóa - Giải trí',
}

print('Dataset folder         :', DATASET_FOLDER)
print('Vietnamese_Embedding   :', VI_MODEL_NAME)
print('PhoBERT encoder        :', PHOBERT_MODEL_NAME)
print('Appendix A - bge-m3    :', BGE_MODEL_NAME, '| RUN_BGE =', RUN_BGE)
print('Appendix B - mpnet     :', MPNET_MODEL_NAME, '| RUN_MPNET =', RUN_MPNET)


---
## Section 1 - Load Dữ Liệu Thô

Đọc toàn bộ **19 file parquet** từ thư mục Dataset.

In [ ]:
if not os.path.isdir(DATASET_FOLDER):
    raise SystemExit(f'Không tìm thấy Dataset: {DATASET_FOLDER}')
expected = set(LABEL_MAP)
found = {f.replace('.parquet', '') for f in os.listdir(DATASET_FOLDER) if f.endswith('.parquet')}
missing = sorted(expected - found)
if missing:
    raise SystemExit(f'Thiếu parquet: {missing}')
for slug in sorted(expected):
    cols = set(pq.read_schema(os.path.join(DATASET_FOLDER, f'{slug}.parquet')).names)
    if not {'title', 'content'}.issubset(cols):
        raise SystemExit(f'{slug}.parquet thiếu cột title/content: {sorted(cols)}')

records = []
load_rows = []
for fname in sorted(os.listdir(DATASET_FOLDER)):
    if not fname.endswith('.parquet'):
        continue
    slug = fname.replace('.parquet', '')
    if slug not in LABEL_MAP:
        continue
    dfc = pd.read_parquet(os.path.join(DATASET_FOLDER, fname)).copy()
    title_missing = int(dfc['title'].isna().sum())
    content_missing = int(dfc['content'].isna().sum())
    title_empty = int(dfc['title'].fillna('').astype(str).str.strip().eq('').sum())
    content_empty = int(dfc['content'].fillna('').astype(str).str.strip().eq('').sum())
    both_empty = int((dfc['title'].fillna('').astype(str).str.strip().eq('') & dfc['content'].fillna('').astype(str).str.strip().eq('')).sum())
    load_rows.append({
        'file': fname,
        'label': LABEL_MAP[slug],
        'rows': int(len(dfc)),
        'title_missing': title_missing,
        'content_missing': content_missing,
        'title_empty': title_empty,
        'content_empty': content_empty,
        'both_empty': both_empty,
    })
    dfc['label'] = LABEL_MAP[slug]
    dfc['source_file'] = fname
    records.append(dfc[['title', 'content', 'label', 'source_file']])

load_df = pd.DataFrame(load_rows).sort_values('rows', ascending=False)
df_raw = pd.concat(records, ignore_index=True)
raw_total = int(len(df_raw))
df_raw['title'] = df_raw['title'].fillna('').astype(str).str.strip()
df_raw['content'] = df_raw['content'].fillna('').astype(str).str.strip()
removed_both_empty = int(((df_raw['title'] == '') & (df_raw['content'] == '')).sum())
df_raw = df_raw[(df_raw['title'] != '') | (df_raw['content'] != '')].copy()
df_raw['full_text'] = (df_raw['title'] + ' ' + df_raw['content']).str.strip()
df_raw['text_len'] = df_raw['full_text'].str.split().str.len()
df_raw = df_raw.reset_index(drop=True)
label_counts = df_raw['label'].value_counts().sort_values(ascending=False)

print('=' * 80)
print('TỔNG QUAN LOAD DỮ LIỆU')
print('=' * 80)
print('Tổng số dòng thô        :', f'{raw_total:,}')
print('Số dòng bị loại (rỗng)  :', f'{removed_both_empty:,}')
print('Số mẫu sau khi lọc      :', f'{len(df_raw):,}')
print('Số chủ đề               :', df_raw['label'].nunique())
print('Thiếu title (NaN)       :', f"{int(load_df['title_missing'].sum()):,}")
print('Thiếu content (NaN)     :', f"{int(load_df['content_missing'].sum()):,}")
print('Title rỗng              :', f"{int(load_df['title_empty'].sum()):,}")
print('Content rỗng            :', f"{int(load_df['content_empty'].sum()):,}")
print('Cả title+content rỗng   :', f"{int(load_df['both_empty'].sum()):,}")
print()
print('SỐ LƯỢNG THEO CHỦ ĐỀ')
print(label_counts.to_string())
print()
print('CHI TIẾT THEO FILE / CHỦ ĐỀ')
print(load_df.to_string(index=False))


---
## Section 2 - Khám Phá Dữ Liệu (EDA)

Phân tích phân bố class và độ dài văn bản trước khi encode embedding.

In [ ]:
vc = df_raw['label'].value_counts().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
axes[0].bar(vc.index, vc.values, color=['#d95f02' if v < vc.median() * 0.75 else '#1b9e77' for v in vc.values])
axes[0].set_title('Phân bố số bài theo chủ đề')
axes[0].tick_params(axis='x', rotation=65)
axes[0].set_ylabel('Số bài')
sns.histplot(df_raw['text_len'], bins=80, kde=True, color='#457b9d', ax=axes[1])
axes[1].set_title('Độ dài full_text')
axes[1].set_xlabel('Số từ')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '01_eda_overview.png'), bbox_inches='tight')
plt.show()

---
## Section 3 - Tiền Xử Lý Văn Bản

Dùng `title + content`, làm sạch nhẹ, **không loại stopwords** để giữ ngữ nghĩa cho embedding retrieval.

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^\w\sÀ-ỹ]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return ''
    text = ViTokenizer.tokenize(text)
    return re.sub(r'\s+', ' ', text).strip()

if os.path.exists(PROCESSED_DATA_PATH):
    print(f'[OK] Cache tồn tại: {PROCESSED_DATA_PATH}')
else:
    tqdm.pandas()
    df_proc = df_raw[['title', 'content', 'full_text', 'label', 'source_file']].copy()
    with timer('Tiền xử lý full_text'):
        df_proc['clean_text'] = df_proc['full_text'].progress_apply(normalize_text)
    df_proc = df_proc[df_proc['clean_text'] != ''].reset_index(drop=True)
    classes = sorted(df_proc['label'].unique())
    label2id = {label: idx for idx, label in enumerate(classes)}
    id2label = {idx: label for label, idx in label2id.items()}
    with open(PROCESSED_DATA_PATH, 'wb') as f:
        pickle.dump({'df': df_proc, 'label2id': label2id, 'id2label': id2label}, f)
with open(PROCESSED_DATA_PATH, 'rb') as f:
    cache = pickle.load(f)
df = cache['df']
label2id = cache['label2id']
id2label = cache['id2label']
classes = [id2label[i] for i in range(len(id2label))]
N_CLASSES = len(classes)


---
## Section 4 - Chia Train/Test

Chia dữ liệu theo `stratify` để cả 3 hướng embedding dùng chung một split công bằng.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df['label'],
)

train_meta = pd.DataFrame({'text': X_train_raw.reset_index(drop=True), 'label': y_train.reset_index(drop=True)})
test_meta = pd.DataFrame({'text': X_test_raw.reset_index(drop=True), 'label': y_test.reset_index(drop=True)})
train_label_ids = train_meta['label'].map(label2id).to_numpy()
test_label_ids = test_meta['label'].map(label2id).to_numpy()

_split_summary = pd.DataFrame({
    'train': train_meta['label'].value_counts().reindex(classes, fill_value=0),
    'test': test_meta['label'].value_counts().reindex(classes, fill_value=0),
})
_split_summary['total'] = _split_summary['train'] + _split_summary['test']

print('Train:', f'{len(train_meta):,}', '| Test:', f'{len(test_meta):,}')
print('Train ratio:', f'{len(train_meta) / len(df):.2%}', '| Test ratio:', f'{len(test_meta) / len(df):.2%}')
print(_split_summary.to_string())

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, model_name, max_length, batch_size):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    vectors = []
    num_batches = len(range(0, len(texts), batch_size))
    print(f'[ENCODE] model={model_name} | num_texts={len(texts):,} | batch_size={batch_size} | max_length={max_length} | num_batches={num_batches:,}')
    for start in tqdm(range(0, len(texts), batch_size), desc=f'Encoding {model_name}'):
        batch = texts[start:start + batch_size]
        half = (max_length - 2) // 2
        batch_ids = []
        batch_masks = []
        for text in batch:
            token_ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
            if len(token_ids) > (max_length - 2):
                token_ids = token_ids[:half] + token_ids[-half:]
            encoded_single = tokenizer.prepare_for_model(
                token_ids,
                add_special_tokens=True,
                truncation=False,
                max_length=max_length,
                padding='max_length',
                return_attention_mask=True,
            )
            batch_ids.append(encoded_single['input_ids'])
            batch_masks.append(encoded_single['attention_mask'])
        encoded = {
            'input_ids': torch.tensor(batch_ids, dtype=torch.long),
            'attention_mask': torch.tensor(batch_masks, dtype=torch.long),
        }
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
            pooled = outputs.pooler_output if getattr(outputs, 'pooler_output', None) is not None else mean_pool(outputs.last_hidden_state, encoded['attention_mask'])
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        vectors.append(pooled.cpu().numpy().astype(np.float32))
    del model
    torch.cuda.empty_cache()
    return np.vstack(vectors)

def run_semantic_knn(train_embeddings, test_embeddings, train_label_ids, test_label_ids, top_k, vote_power, min_similarity):
    index = NearestNeighbors(n_neighbors=min(top_k, len(train_embeddings)), metric='cosine', algorithm='brute')
    index.fit(train_embeddings)
    topk_distances, topk_indices = index.kneighbors(test_embeddings, return_distance=True)
    topk_similarities = np.clip(1.0 - topk_distances, 0.0, 1.0)
    topk_weights = np.where(topk_similarities >= min_similarity, np.power(np.clip(topk_similarities, 1e-6, 1.0), vote_power), 0.0)
    raw_scores = np.zeros((len(test_embeddings), N_CLASSES), dtype=np.float32)
    for row_idx in range(len(test_embeddings)):
        for nbr_pos, train_idx in enumerate(topk_indices[row_idx]):
            raw_scores[row_idx, train_label_ids[train_idx]] += topk_weights[row_idx, nbr_pos]
    y_pred_ids = raw_scores.argmax(axis=1)
    return {'topk_indices': topk_indices, 'topk_similarities': topk_similarities, 'raw_scores': raw_scores, 'y_true_ids': test_label_ids, 'y_pred_ids': y_pred_ids, 'acc': accuracy_score(test_label_ids, y_pred_ids), 'f1w': f1_score(test_label_ids, y_pred_ids, average='weighted'), 'f1m': f1_score(test_label_ids, y_pred_ids, average='macro')}

def run_embedding_benchmarks(train_embeddings, test_embeddings, train_label_ids, test_label_ids, top_k, vote_power, min_similarity):
    benchmark_results = {}
    with timer('Fit + predict - Semantic KNN'):
        knn_res = run_semantic_knn(train_embeddings, test_embeddings, train_label_ids, test_label_ids, top_k, vote_power, min_similarity)
    benchmark_results['KNN'] = {'model_name': 'Semantic KNN', 'y_pred_ids': knn_res['y_pred_ids'], 'acc': knn_res['acc'], 'f1w': knn_res['f1w'], 'f1m': knn_res['f1m'], 'details': knn_res}

    lr_model = make_pipeline(StandardScaler(), LogisticRegression(C=4.0, max_iter=2000, class_weight='balanced', solver='lbfgs'))
    with timer('Fit + predict - Logistic Regression'):
        lr_model.fit(train_embeddings, train_label_ids)
        lr_pred = lr_model.predict(test_embeddings)
    benchmark_results['LR'] = {'model_name': 'Logistic Regression', 'y_pred_ids': lr_pred, 'acc': accuracy_score(test_label_ids, lr_pred), 'f1w': f1_score(test_label_ids, lr_pred, average='weighted'), 'f1m': f1_score(test_label_ids, lr_pred, average='macro'), 'details': {'estimator': lr_model}}

    svm_model = make_pipeline(StandardScaler(), LinearSVC(C=1.0, class_weight='balanced', max_iter=5000))
    with timer('Fit + predict - Linear SVM'):
        svm_model.fit(train_embeddings, train_label_ids)
        svm_pred = svm_model.predict(test_embeddings)
    benchmark_results['SVM'] = {'model_name': 'Linear SVM', 'y_pred_ids': svm_pred, 'acc': accuracy_score(test_label_ids, svm_pred), 'f1w': f1_score(test_label_ids, svm_pred, average='weighted'), 'f1m': f1_score(test_label_ids, svm_pred, average='macro'), 'details': {'estimator': svm_model}}

    mlp_model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(512, 256), activation='relu', solver='adam', alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=80, early_stopping=True, n_iter_no_change=8, validation_fraction=0.1, random_state=SEED))
    with timer('Fit + predict - MLP'):
        mlp_model.fit(train_embeddings, train_label_ids)
        mlp_pred = mlp_model.predict(test_embeddings)
    benchmark_results['MLP'] = {'model_name': 'MLP', 'y_pred_ids': mlp_pred, 'acc': accuracy_score(test_label_ids, mlp_pred), 'f1w': f1_score(test_label_ids, mlp_pred, average='weighted'), 'f1m': f1_score(test_label_ids, mlp_pred, average='macro'), 'details': {'estimator': mlp_model}}

    summary_df = pd.DataFrame([{'model': v['model_name'], 'accuracy': v['acc'], 'f1_weighted': v['f1w'], 'f1_macro': v['f1m']} for v in benchmark_results.values()]).sort_values(['f1_macro', 'f1_weighted', 'accuracy'], ascending=False).reset_index(drop=True)
    return benchmark_results, summary_df


---
## Section 5 - Vietnamese_Embedding: Encode Embedding


In [ ]:
if os.path.exists(VI_TRAIN_EMB_PATH) and os.path.exists(VI_TEST_EMB_PATH):
    log('Đã có cache Vietnamese_Embedding train/test - bỏ qua encode', 'OK')
    vi_train_embeddings = np.load(VI_TRAIN_EMB_PATH, allow_pickle=True)['embeddings']
    vi_test_embeddings = np.load(VI_TEST_EMB_PATH, allow_pickle=True)['embeddings']
else:
    with timer('Encode train embeddings - Vietnamese_Embedding'):
        vi_train_embeddings = encode_texts(train_meta['text'].tolist(), VI_MODEL_NAME, VI_MAX_LENGTH, VI_BATCH_SIZE)
    np.savez_compressed(VI_TRAIN_EMB_PATH, embeddings=vi_train_embeddings)
    with timer('Encode test embeddings - Vietnamese_Embedding'):
        vi_test_embeddings = encode_texts(test_meta['text'].tolist(), VI_MODEL_NAME, VI_MAX_LENGTH, VI_BATCH_SIZE)
    np.savez_compressed(VI_TEST_EMB_PATH, embeddings=vi_test_embeddings)
print('vi_train_embeddings:', vi_train_embeddings.shape)
print('vi_test_embeddings :', vi_test_embeddings.shape)


---
## Section 6 - Vietnamese_Embedding: Benchmark 4 Mô Hình


In [ ]:
vi_benchmark_results, vi_summary_df = run_embedding_benchmarks(vi_train_embeddings, vi_test_embeddings, train_label_ids, test_label_ids, VI_TOP_K, VI_VOTE_POWER, VI_MIN_SIMILARITY)
print(vi_summary_df.to_string(index=False))
_vi_key_map = {'Semantic KNN': 'KNN', 'Logistic Regression': 'LR', 'Linear SVM': 'SVM', 'MLP': 'MLP'}
vi_best_model_key = _vi_key_map[vi_summary_df.iloc[0]['model']]
vi_best_model_name = vi_benchmark_results[vi_best_model_key]['model_name']
vi_best_y_pred_ids = vi_benchmark_results[vi_best_model_key]['y_pred_ids']
print(f'Best model theo F1-macro: {vi_best_model_name}')


---
## Section 7 - Vietnamese_Embedding: Đánh Giá So Sánh


In [ ]:
with open(os.path.join(RESULTS_DIR, 'vi_embedding_model_comparison.csv'), 'w', encoding='utf-8-sig') as f:
    vi_summary_df.to_csv(f, index=False)
print(vi_summary_df.to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=vi_summary_df, x='model', y='accuracy', ax=axes[0], color='#4c78a8')
axes[0].set_title('Accuracy'); axes[0].tick_params(axis='x', rotation=20)
sns.barplot(data=vi_summary_df, x='model', y='f1_weighted', ax=axes[1], color='#f58518')
axes[1].set_title('F1-weighted'); axes[1].tick_params(axis='x', rotation=20)
sns.barplot(data=vi_summary_df, x='model', y='f1_macro', ax=axes[2], color='#54a24b')
axes[2].set_title('F1-macro'); axes[2].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '02_vi_model_comparison.png'), bbox_inches='tight')
plt.show()
for key, item in vi_benchmark_results.items():
    print('\n' + '=' * 80)
    print(item['model_name'])
    print('=' * 80)
    report_str = classification_report(test_label_ids, item['y_pred_ids'], target_names=classes, digits=4)
    print(report_str)
vi_best_cm = confusion_matrix(test_label_ids, vi_best_y_pred_ids, labels=list(range(N_CLASSES)))
vi_best_report_dict = classification_report(test_label_ids, vi_best_y_pred_ids, target_names=classes, output_dict=True, zero_division=0)
vi_best_diag_df = pd.DataFrame([{'class': cls, 'precision': vi_best_report_dict[cls]['precision'], 'recall': vi_best_report_dict[cls]['recall'], 'f1': vi_best_report_dict[cls]['f1-score'], 'support': int(vi_best_report_dict[cls]['support'])} for cls in classes]).sort_values('f1')
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(vi_best_cm, cmap='Blues', ax=axes[0], xticklabels=classes, yticklabels=classes)
axes[0].set_title(f'{vi_best_model_name} - Confusion Matrix'); axes[0].tick_params(axis='x', rotation=90); axes[0].tick_params(axis='y', rotation=0)
sns.barplot(data=vi_best_diag_df, y='class', x='f1', color='#4c78a8', ax=axes[1])
axes[1].set_xlim(0, 1); axes[1].set_title(f'{vi_best_model_name} - F1 theo l?p')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '03_vi_best_model_eval.png'), bbox_inches='tight')
plt.show()


---
## Section 8 - Vietnamese_Embedding: Phân Tích Lỗi Của Mô Hình Tốt Nhất


In [ ]:
vi_best_error_indices = np.where(test_label_ids != vi_best_y_pred_ids)[0]
print('Best model:', vi_best_model_name)
print('S? m?u d? ?o?n sai:', len(vi_best_error_indices))
vi_error_rows = []
for idx in vi_best_error_indices[:15]:
    row = {'sample_idx': int(idx), 'true': id2label[test_label_ids[idx]], 'pred': id2label[vi_best_y_pred_ids[idx]]}
    if vi_best_model_key == 'KNN':
        row['top1_sim'] = round(float(vi_benchmark_results['KNN']['details']['topk_similarities'][idx, 0]), 4)
        row['neighbor_labels'] = [train_meta.iloc[n_idx]['label'] for n_idx in vi_benchmark_results['KNN']['details']['topk_indices'][idx]]
    vi_error_rows.append(row)
print(pd.DataFrame(vi_error_rows).to_string(index=False, max_colwidth=120))


---
## Section 9 - Vietnamese_Embedding: Export


In [ ]:
vi_label_config = {
    'embedding_model_name': VI_MODEL_NAME,
    'strategy': 'embedding_benchmark',
    'text_representation': 'title + content | head-tail dynamic',
    'max_length': VI_MAX_LENGTH,
    'n_classes': N_CLASSES,
    'classes': classes,
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},
    'benchmarks': ['Semantic KNN', 'Logistic Regression', 'Linear SVM', 'MLP'],
    'top_k': VI_TOP_K, 'vote_power': VI_VOTE_POWER, 'min_similarity': VI_MIN_SIMILARITY, 'metric': 'cosine',
}
with open(VI_LABEL_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(vi_label_config, f, ensure_ascii=False, indent=2)
with open(VI_RETRIEVAL_CACHE_PATH, 'wb') as f:
    pickle.dump({'embedding_model_name': VI_MODEL_NAME, 'model_comparison': vi_summary_df.to_dict(orient='records'), 'best_model': vi_best_model_name}, f)
print('Đã lưu:', VI_LABEL_CONFIG_PATH)
print('Đã lưu:', VI_RETRIEVAL_CACHE_PATH)


---
## Section 10 - PhoBERT Encoder: Encode Embedding


In [ ]:
if os.path.exists(PHOBERT_TRAIN_EMB_PATH) and os.path.exists(PHOBERT_TEST_EMB_PATH):
    log('Đã có cache PhoBERT encoder train/test - bỏ qua encode', 'OK')
    phobert_train_embeddings = np.load(PHOBERT_TRAIN_EMB_PATH, allow_pickle=True)['embeddings']
    phobert_test_embeddings = np.load(PHOBERT_TEST_EMB_PATH, allow_pickle=True)['embeddings']
else:
    with timer('Encode train embeddings - PhoBERT encoder'):
        phobert_train_embeddings = encode_texts(train_meta['text'].tolist(), PHOBERT_MODEL_NAME, PHOBERT_MAX_LENGTH, PHOBERT_BATCH_SIZE)
    np.savez_compressed(PHOBERT_TRAIN_EMB_PATH, embeddings=phobert_train_embeddings)
    with timer('Encode test embeddings - PhoBERT encoder'):
        phobert_test_embeddings = encode_texts(test_meta['text'].tolist(), PHOBERT_MODEL_NAME, PHOBERT_MAX_LENGTH, PHOBERT_BATCH_SIZE)
    np.savez_compressed(PHOBERT_TEST_EMB_PATH, embeddings=phobert_test_embeddings)
print('phobert_train_embeddings:', phobert_train_embeddings.shape)
print('phobert_test_embeddings :', phobert_test_embeddings.shape)


---
## Section 11 - PhoBERT Encoder: Benchmark 4 Mô Hình


In [ ]:
phobert_benchmark_results, phobert_summary_df = run_embedding_benchmarks(phobert_train_embeddings, phobert_test_embeddings, train_label_ids, test_label_ids, PHOBERT_TOP_K, PHOBERT_VOTE_POWER, PHOBERT_MIN_SIMILARITY)
print(phobert_summary_df.to_string(index=False))
_phobert_key_map = {'Semantic KNN': 'KNN', 'Logistic Regression': 'LR', 'Linear SVM': 'SVM', 'MLP': 'MLP'}
phobert_best_model_key = _phobert_key_map[phobert_summary_df.iloc[0]['model']]
phobert_best_model_name = phobert_benchmark_results[phobert_best_model_key]['model_name']
phobert_best_y_pred_ids = phobert_benchmark_results[phobert_best_model_key]['y_pred_ids']
print(f'Best model theo F1-macro: {phobert_best_model_name}')


---
## Section 12 - PhoBERT Encoder: Đánh Giá So Sánh


In [ ]:
with open(os.path.join(RESULTS_DIR, 'phobert_embedding_model_comparison.csv'), 'w', encoding='utf-8-sig') as f:
    phobert_summary_df.to_csv(f, index=False)
print(phobert_summary_df.to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=phobert_summary_df, x='model', y='accuracy', ax=axes[0], color='#355070')
axes[0].set_title('Accuracy'); axes[0].tick_params(axis='x', rotation=20)
sns.barplot(data=phobert_summary_df, x='model', y='f1_weighted', ax=axes[1], color='#6d597a')
axes[1].set_title('F1-weighted'); axes[1].tick_params(axis='x', rotation=20)
sns.barplot(data=phobert_summary_df, x='model', y='f1_macro', ax=axes[2], color='#b56576')
axes[2].set_title('F1-macro'); axes[2].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_phobert_model_comparison.png'), bbox_inches='tight')
plt.show()
for key, item in phobert_benchmark_results.items():
    print('\n' + '=' * 80)
    print(item['model_name'])
    print('=' * 80)
    report_str = classification_report(test_label_ids, item['y_pred_ids'], target_names=classes, digits=4)
    print(report_str)
phobert_best_cm = confusion_matrix(test_label_ids, phobert_best_y_pred_ids, labels=list(range(N_CLASSES)))
phobert_best_report_dict = classification_report(test_label_ids, phobert_best_y_pred_ids, target_names=classes, output_dict=True, zero_division=0)
phobert_best_diag_df = pd.DataFrame([{'class': cls, 'precision': phobert_best_report_dict[cls]['precision'], 'recall': phobert_best_report_dict[cls]['recall'], 'f1': phobert_best_report_dict[cls]['f1-score'], 'support': int(phobert_best_report_dict[cls]['support'])} for cls in classes]).sort_values('f1')
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(phobert_best_cm, cmap='Purples', ax=axes[0], xticklabels=classes, yticklabels=classes)
axes[0].set_title(f'{phobert_best_model_name} - Confusion Matrix'); axes[0].tick_params(axis='x', rotation=90); axes[0].tick_params(axis='y', rotation=0)
sns.barplot(data=phobert_best_diag_df, y='class', x='f1', color='#6d597a', ax=axes[1])
axes[1].set_xlim(0, 1); axes[1].set_title(f'{phobert_best_model_name} - F1 theo l?p')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '05_phobert_best_model_eval.png'), bbox_inches='tight')
plt.show()


---
## Section 13 - PhoBERT Encoder: Phân Tích Lỗi Của Mô Hình Tốt Nhất


In [ ]:
phobert_best_error_indices = np.where(test_label_ids != phobert_best_y_pred_ids)[0]
print('Best model:', phobert_best_model_name)
print('S? m?u d? ?o?n sai:', len(phobert_best_error_indices))
phobert_error_rows = []
for idx in phobert_best_error_indices[:15]:
    row = {'sample_idx': int(idx), 'true': id2label[test_label_ids[idx]], 'pred': id2label[phobert_best_y_pred_ids[idx]]}
    if phobert_best_model_key == 'KNN':
        row['top1_sim'] = round(float(phobert_benchmark_results['KNN']['details']['topk_similarities'][idx, 0]), 4)
        row['neighbor_labels'] = [train_meta.iloc[n_idx]['label'] for n_idx in phobert_benchmark_results['KNN']['details']['topk_indices'][idx]]
    phobert_error_rows.append(row)
print(pd.DataFrame(phobert_error_rows).to_string(index=False, max_colwidth=120))


---
## Section 14 - PhoBERT Encoder: Export


In [ ]:
phobert_label_config = {
    'embedding_model_name': PHOBERT_MODEL_NAME,
    'strategy': 'embedding_benchmark',
    'text_representation': 'title + content | head-tail dynamic',
    'max_length': PHOBERT_MAX_LENGTH,
    'n_classes': N_CLASSES,
    'classes': classes,
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},
    'benchmarks': ['Semantic KNN', 'Logistic Regression', 'Linear SVM', 'MLP'],
    'top_k': PHOBERT_TOP_K, 'vote_power': PHOBERT_VOTE_POWER, 'min_similarity': PHOBERT_MIN_SIMILARITY, 'metric': 'cosine',
}
with open(PHOBERT_LABEL_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(phobert_label_config, f, ensure_ascii=False, indent=2)
with open(PHOBERT_RETRIEVAL_CACHE_PATH, 'wb') as f:
    pickle.dump({'embedding_model_name': PHOBERT_MODEL_NAME, 'model_comparison': phobert_summary_df.to_dict(orient='records'), 'best_model': phobert_best_model_name}, f)
print('Đã lưu:', PHOBERT_LABEL_CONFIG_PATH)
print('Đã lưu:', PHOBERT_RETRIEVAL_CACHE_PATH)


---
## Section 15 - Appendix A: bge-m3 Encode Embedding

Code sẵn nhưng mặc định không chạy. Đổi `RUN_BGE = True` ở phần config nếu muốn bật nhánh này.

In [ ]:
if not RUN_BGE:
    bge_train_embeddings = None
    bge_test_embeddings = None
    print('SKIP Appendix A - bge-m3 vì RUN_BGE = False')
elif os.path.exists(BGE_TRAIN_EMB_PATH) and os.path.exists(BGE_TEST_EMB_PATH):
    log('Đã có cache bge-m3 train/test - bỏ qua encode', 'OK')
    bge_train_embeddings = np.load(BGE_TRAIN_EMB_PATH, allow_pickle=True)['embeddings']
    bge_test_embeddings = np.load(BGE_TEST_EMB_PATH, allow_pickle=True)['embeddings']
else:
    with timer('Encode train embeddings - bge-m3'):
        bge_train_embeddings = encode_texts(train_meta['text'].tolist(), BGE_MODEL_NAME, BGE_MAX_LENGTH, BGE_BATCH_SIZE)
    np.savez_compressed(BGE_TRAIN_EMB_PATH, embeddings=bge_train_embeddings)
    with timer('Encode test embeddings - bge-m3'):
        bge_test_embeddings = encode_texts(test_meta['text'].tolist(), BGE_MODEL_NAME, BGE_MAX_LENGTH, BGE_BATCH_SIZE)
    np.savez_compressed(BGE_TEST_EMB_PATH, embeddings=bge_test_embeddings)
if RUN_BGE and bge_train_embeddings is not None:
    print('bge_train_embeddings:', bge_train_embeddings.shape)
    print('bge_test_embeddings :', bge_test_embeddings.shape)


---
## Section 16 - Appendix A: bge-m3 Benchmark 4 Mô Hình


In [ ]:
if not RUN_BGE:
    bge_benchmark_results = None
    bge_summary_df = None
    print('SKIP Appendix A benchmark vì RUN_BGE = False')
else:
    bge_benchmark_results, bge_summary_df = run_embedding_benchmarks(bge_train_embeddings, bge_test_embeddings, train_label_ids, test_label_ids, BGE_TOP_K, BGE_VOTE_POWER, BGE_MIN_SIMILARITY)
    print(bge_summary_df.to_string(index=False))


---
## Section 17 - Appendix A: bge-m3 Đánh Giá So Sánh


In [ ]:
if not RUN_BGE:
    print('SKIP Appendix A compare vì RUN_BGE = False')
else:
    with open(os.path.join(RESULTS_DIR, 'bge_embedding_model_comparison.csv'), 'w', encoding='utf-8-sig') as f:
        bge_summary_df.to_csv(f, index=False)
    print(bge_summary_df.to_string(index=False))


---
## Section 18 - Appendix A: bge-m3 Export


In [ ]:
if not RUN_BGE:
    print('SKIP Appendix A export vì RUN_BGE = False')
else:
    bge_label_config = {'embedding_model_name': BGE_MODEL_NAME, 'strategy': 'embedding_benchmark', 'text_representation': 'title + content | head-tail dynamic', 'max_length': BGE_MAX_LENGTH, 'n_classes': N_CLASSES, 'classes': classes, 'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}, 'benchmarks': ['Semantic KNN', 'Logistic Regression', 'Linear SVM', 'MLP'], 'top_k': BGE_TOP_K, 'vote_power': BGE_VOTE_POWER, 'min_similarity': BGE_MIN_SIMILARITY, 'metric': 'cosine'}
    with open(BGE_LABEL_CONFIG_PATH, 'w', encoding='utf-8') as f:
        json.dump(bge_label_config, f, ensure_ascii=False, indent=2)
    with open(BGE_RETRIEVAL_CACHE_PATH, 'wb') as f:
        pickle.dump({'embedding_model_name': BGE_MODEL_NAME, 'model_comparison': bge_summary_df.to_dict(orient='records')}, f)
    print('Đã lưu:', BGE_LABEL_CONFIG_PATH)
    print('Đã lưu:', BGE_RETRIEVAL_CACHE_PATH)


---
## Section 19 - Appendix B: Multilingual mpnet Encode Embedding

Code sẵn nhưng mặc định không chạy. Đổi `RUN_MPNET = True` ở phần config nếu muốn bật nhánh này.

In [ ]:
if not RUN_MPNET:
    mpnet_train_embeddings = None
    mpnet_test_embeddings = None
    print('SKIP Appendix B - mpnet vì RUN_MPNET = False')
elif os.path.exists(MPNET_TRAIN_EMB_PATH) and os.path.exists(MPNET_TEST_EMB_PATH):
    log('Đã có cache mpnet train/test - bỏ qua encode', 'OK')
    mpnet_train_embeddings = np.load(MPNET_TRAIN_EMB_PATH, allow_pickle=True)['embeddings']
    mpnet_test_embeddings = np.load(MPNET_TEST_EMB_PATH, allow_pickle=True)['embeddings']
else:
    with timer('Encode train embeddings - mpnet multilingual'):
        mpnet_train_embeddings = encode_texts(train_meta['text'].tolist(), MPNET_MODEL_NAME, MPNET_MAX_LENGTH, MPNET_BATCH_SIZE)
    np.savez_compressed(MPNET_TRAIN_EMB_PATH, embeddings=mpnet_train_embeddings)
    with timer('Encode test embeddings - mpnet multilingual'):
        mpnet_test_embeddings = encode_texts(test_meta['text'].tolist(), MPNET_MODEL_NAME, MPNET_MAX_LENGTH, MPNET_BATCH_SIZE)
    np.savez_compressed(MPNET_TEST_EMB_PATH, embeddings=mpnet_test_embeddings)
if RUN_MPNET and mpnet_train_embeddings is not None:
    print('mpnet_train_embeddings:', mpnet_train_embeddings.shape)
    print('mpnet_test_embeddings :', mpnet_test_embeddings.shape)


---
## Section 20 - Appendix B: Multilingual mpnet Benchmark 4 Mô Hình


In [ ]:
if not RUN_MPNET:
    mpnet_benchmark_results = None
    mpnet_summary_df = None
    print('SKIP Appendix B benchmark vì RUN_MPNET = False')
else:
    mpnet_benchmark_results, mpnet_summary_df = run_embedding_benchmarks(mpnet_train_embeddings, mpnet_test_embeddings, train_label_ids, test_label_ids, MPNET_TOP_K, MPNET_VOTE_POWER, MPNET_MIN_SIMILARITY)
    print(mpnet_summary_df.to_string(index=False))


---
## Section 21 - Appendix B: Multilingual mpnet Đánh Giá So Sánh


In [ ]:
if not RUN_MPNET:
    print('SKIP Appendix B compare vì RUN_MPNET = False')
else:
    with open(os.path.join(RESULTS_DIR, 'mpnet_embedding_model_comparison.csv'), 'w', encoding='utf-8-sig') as f:
        mpnet_summary_df.to_csv(f, index=False)
    print(mpnet_summary_df.to_string(index=False))


---
## Section 22 - Appendix B: Multilingual mpnet Export


In [ ]:
if not RUN_MPNET:
    print('SKIP Appendix B export vì RUN_MPNET = False')
else:
    mpnet_label_config = {'embedding_model_name': MPNET_MODEL_NAME, 'strategy': 'embedding_benchmark', 'text_representation': 'title + content | head-tail dynamic', 'max_length': MPNET_MAX_LENGTH, 'n_classes': N_CLASSES, 'classes': classes, 'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}, 'benchmarks': ['Semantic KNN', 'Logistic Regression', 'Linear SVM', 'MLP'], 'top_k': MPNET_TOP_K, 'vote_power': MPNET_VOTE_POWER, 'min_similarity': MPNET_MIN_SIMILARITY, 'metric': 'cosine'}
    with open(MPNET_LABEL_CONFIG_PATH, 'w', encoding='utf-8') as f:
        json.dump(mpnet_label_config, f, ensure_ascii=False, indent=2)
    with open(MPNET_RETRIEVAL_CACHE_PATH, 'wb') as f:
        pickle.dump({'embedding_model_name': MPNET_MODEL_NAME, 'model_comparison': mpnet_summary_df.to_dict(orient='records')}, f)
    print('Đã lưu:', MPNET_LABEL_CONFIG_PATH)
    print('Đã lưu:', MPNET_RETRIEVAL_CACHE_PATH)
